# Study 940 — The Turnover Budget — the teardown

The frequency table, the break-even cost per unit of traded notional, the cost surface and its inversion, the borrow sweep, the era cut, bootstrap Sharpe CIs, paired speed races, the long-only beta check, and the live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `7af2719e8031`, as-of 2026-06-30).

**Construction.** 12-1 signal (252d return, 21d skip) on the eleven Select Sector SPDRs; long top 3 / short bottom 3, equal-weighted, 0.5 gross per side. Signal through day *t* → weights effective *t+1* → trade charged *t+1*: **one lag, and only one**. Weights drift with realised returns between rebalances. Traded notional is `Σ|w_new − w_drifted|` per unit NAV; cost is one-way × NAV; the short leg accrues borrow daily.

**Excess-of-cash convention.** A dollar-neutral book is funded by its own collateral, which earns the bill, so the long-minus-short spread *is* the excess-of-cash return — subtracting BIL again would double-count. BIL gates only the long-only cross-check (from 2007-05-30).

In [1]:
R = {'start': '1999-12-23', 'end': '2026-06-30', 'n_days': 6668, 'fp': '7af2719e8031', 'cost_bps': 5.0, 'borrow_bps': 40.0, 'labels': ['daily', 'weekly', 'monthly', 'quarterly'], 'n_rebal': [6668, 1384, 318, 106], 'turnover': [31.27, 13.44, 5.81, 3.27], 'turn_per_rebal': [0.1241, 0.2569, 0.4828, 0.8143], 'churn_share': [0.9175, 0.919, 0.9228, 0.9299], 'sharpe_gross': [0.121, 0.105, 0.099, 0.013], 'sharpe_net': [-0.096, -0.003, 0.037, -0.034], 'ret_gross': [0.98, 0.85, 0.79, 0.1], 'ret_net': [-0.78, -0.02, 0.3, -0.27], 'vol': [8.13, 8.11, 7.98, 7.83], 'maxdd': [-36.7, -31.7, -26.3, -39.4], 't_gross': [0.67, 0.59, 0.55, 0.07], 'breakeven': [2.51, 4.83, 10.09, -3.24], 'cost_grid': [0.0, 1.0, 2.5, 5.0, 10.0, 25.0], 'cost_surface': [[0.097, 0.08, 0.073, -0.014], [0.058, 0.063, 0.066, -0.018], [0.0, 0.039, 0.055, -0.024], [-0.096, -0.003, 0.037, -0.034], [-0.288, -0.086, 0.001, -0.055], [-0.862, -0.334, -0.109, -0.118]], 'borrow_grid': [0.0, 25.0, 50.0, 100.0], 'borrow_surface': [[-0.071, 0.022, 0.062, -0.008], [-0.087, 0.006, 0.047, -0.025], [-0.102, -0.009, 0.031, -0.041], [-0.133, -0.04, -0.001, -0.074]], 'param_grid': [(2, 252, 21, 0.055, 0.31), (3, 252, 21, 0.099, 0.55), (4, 252, 21, 0.072, 0.39), (5, 252, 21, -0.006, -0.02), (3, 126, 21, 0.02, 0.11), (3, 252, 0, 0.046, 0.25), (3, 63, 5, 0.003, 0.02)], 'ci_gross': [(-0.221, 0.473), (-0.232, 0.457), (-0.252, 0.449), (-0.319, 0.346)], 'ci_net': [(-0.431, 0.252), (-0.337, 0.348), (-0.313, 0.388), (-0.366, 0.297)], 'ci_net_negshare': [71.2, 51.4, 41.4, 58.8], 'races': [('daily - monthly', 0.012, 0.18, -0.144, -1.9), ('daily - quarterly', 0.098, 0.9, -0.072, -0.67), ('monthly - quarterly', 0.086, 1.02, 0.071, 0.84)], 'era_early_gross': [0.035, 0.103, 0.137, 0.013], 'era_early_be': [0.31, 4.92, 15.86, -3.04], 'era_late_gross': [0.154, 0.047, 0.014, -0.022], 'era_late_be': [3.5, 1.24, -1.69, -11.07], 'era_split': '2013-01-01', 'lo_start': '2007-05-30', 'lo_sharpe': [0.524, 0.546, 0.539, 0.535], 'lo_t_cash': [2.56, 2.65, 2.62, 2.62], 'lo_sharpe_ew': [0.556, 0.555, 0.551, 0.584], 'lo_alpha': [-0.62, -0.19, -0.15, -0.44], 'lo_t_ew': [-0.33, -0.1, -0.08, -0.24], 'lo_alpha_gross': [0.67, 0.4, 0.09, -0.31], 'lo_t_ew_gross': [0.35, 0.21, 0.05, -0.17], 'lo_turn': [27.2, 12.44, 5.21, 2.82], 'lo_turn_ew': [1.49, 0.72, 0.37, 0.24], 'syn_planted_sharpe': [5.78, 5.53, 4.87, 3.44], 'syn_planted_t': [17.7, 16.9, 15.0, 10.6], 'syn_planted_gap': 13.91, 'syn_null_sharpe': [-0.03, -0.135, -0.154, 0.025], 'syn_null_mean': -0.048, 'syn_null_sd': 0.232, 'syn_null_max': 0.405, 'crossover_bps': 1.0}

> 💡 **In plain words** — one strategy, four alarm clocks. Everything below asks the same question: what did waking up more often cost, and did it buy anything?

## 1. The frequency table

Base assumptions: **5 bps** per unit traded notional, **40 bps/yr** borrow. Both are assumptions rather than tape, and both are swept below.

In [2]:
hdr = 'speed      rebal  turn/yr  churn%  Sgross   tgross   Snet   ret_g%  ret_n%   vol%   maxDD%   breakeven'
print(hdr); print('-'*len(hdr))
for i, lab in enumerate(R['labels']):
    print('%-9s %6d %8.1f %7.1f %+7.3f %+8.2f %+7.3f %+7.2f %+7.2f %6.2f %8.1f %10.2f bps'
          % (lab, R['n_rebal'][i], R['turnover'][i], R['churn_share'][i]*100,
             R['sharpe_gross'][i], R['t_gross'][i], R['sharpe_net'][i],
             R['ret_gross'][i], R['ret_net'][i], R['vol'][i], R['maxdd'][i],
             R['breakeven'][i]))

speed      rebal  turn/yr  churn%  Sgross   tgross   Snet   ret_g%  ret_n%   vol%   maxDD%   breakeven
------------------------------------------------------------------------------------------------------
daily       6668     31.3    91.8  +0.121    +0.67  -0.096   +0.98   -0.78   8.13    -36.7       2.51 bps
weekly      1384     13.4    91.9  +0.105    +0.59  -0.003   +0.85   -0.02   8.11    -31.7       4.83 bps
monthly      318      5.8    92.3  +0.099    +0.55  +0.037   +0.79   +0.30   7.98    -26.3      10.09 bps
quarterly    106      3.3    93.0  +0.013    +0.07  -0.034   +0.10   -0.27   7.83    -39.4      -3.24 bps


Three things to read off it.

1. **Turnover is monotone in the clock** (31.3× → 13.4× → 5.8× → 3.3× NAV per year) — arithmetic, not a finding.
2. **Gross return is *weakly* monotone too** (+0.98% → +0.85% → +0.79% → +0.10%), which is the mechanism the folk theorem posits — but the whole spread from daily to quarterly is a gross Sharpe gap of +0.098 at *t* = +0.90.
3. **92-93% of traded notional is rank churn**, not weight drift. The book is not paying to re-equalise positions it already holds; it is paying because the ranking genuinely changes. A no-trade band would recover very little.

> 💡 **In plain words** — the fast clock does earn a bit more before costs, in the direction the theory predicts. It is just far too small to matter, and the extra trading is real.

## 2. The break-even cost — the study's actual deliverable

Solve `gross_mean − borrow − c × 1e-4 × mean(traded) = 0` for `c`. Borrow stays charged (a holding cost, not a trading cost), so `c` is purely the price of execution the arm can afford, per unit of notional it turns over.

In [3]:
print('speed       gross ret/yr   turnover/yr   break-even cost per unit traded')
for i, lab in enumerate(R['labels']):
    print('%-10s %+12.2f%% %13.1fx %20.2f bps'
          % (lab, R['ret_gross'][i], R['turnover'][i], R['breakeven'][i]))
print('\nquarterly is negative: it loses money BEFORE a single trade is charged,')
print('so no execution standard, however good, makes it viable.')

speed       gross ret/yr   turnover/yr   break-even cost per unit traded
daily             +0.98%          31.3x                 2.51 bps
weekly            +0.85%          13.4x                 4.83 bps
monthly           +0.79%           5.8x                10.09 bps
quarterly         +0.10%           3.3x                -3.24 bps

quarterly is negative: it loses money BEFORE a single trade is charged,
so no execution standard, however good, makes it viable.


## 3. The cost surface — and where the ranking inverts

Net excess-of-cash Sharpe, borrow held at 40 bps/yr.

In [4]:
print('cost/unit |' + ''.join('%10s' % l for l in R['labels']))
print('-'*(11+10*len(R['labels'])))
for c, row in zip(R['cost_grid'], R['cost_surface']):
    print('%7.1f   |' % c + ''.join('%+10.3f' % v for v in row))
best = [R['labels'][max(range(4), key=lambda j: row[j])] for row in R['cost_surface']]
print('\nbest arm by cost level:', dict(zip(R['cost_grid'], best)))

cost/unit |     daily    weekly   monthly quarterly
---------------------------------------------------
    0.0   |    +0.097    +0.080    +0.073    -0.014
    1.0   |    +0.058    +0.063    +0.066    -0.018
    2.5   |    +0.000    +0.039    +0.055    -0.024
    5.0   |    -0.096    -0.003    +0.037    -0.034
   10.0   |    -0.288    -0.086    +0.001    -0.055
   25.0   |    -0.862    -0.334    -0.109    -0.118

best arm by cost level: {0.0: 'daily', 1.0: 'monthly', 2.5: 'monthly', 5.0: 'monthly', 10.0: 'monthly', 25.0: 'monthly'}


The best arm at 0 bps is **daily**; at **1 bp** it is already third, and from there down every row belongs to **monthly** while daily goes to dead last. The inversion is complete **inside the first basis point per unit traded notional**. Since no honest ETF book clears 1 bp all-in while turning over 31× NAV a year, the practical answer is 'slower' — but it is an answer about execution, not about the market.

> 💡 **In plain words** — whether daily or monthly 'wins' depends entirely on what you assume you pay to trade. That is a warning about every rebalance-frequency study you have ever read, including this one.

## 4. Borrow sweep — the second non-tape input

Borrow is a *holding* cost on the short notional, so it lands almost equally on every speed and cannot rescue or condemn one. Reported here so it is swept rather than asserted.

In [5]:
print('borrow/yr |' + ''.join('%10s' % l for l in R['labels']))
for b, row in zip(R['borrow_grid'], R['borrow_surface']):
    print('%7.1f   |' % b + ''.join('%+10.3f' % v for v in row))
print('\nat ZERO borrow the best arm is still only %+.3f' % max(R['borrow_surface'][0]))

borrow/yr |     daily    weekly   monthly quarterly
    0.0   |    -0.071    +0.022    +0.062    -0.008
   25.0   |    -0.087    +0.006    +0.047    -0.025
   50.0   |    -0.102    -0.009    +0.031    -0.041
  100.0   |    -0.133    -0.040    -0.001    -0.074

at ZERO borrow the best arm is still only +0.062


## 4b. Parameter neighbourhood — was the published sort cherry-picked?

The sort is the Jegadeesh-Titman convention (252-day lookback, 21-day skip, top/bottom 3) and it was fixed before the tape was run. Here is the neighbourhood around it on the monthly clock, gross of everything, so nobody has to take that on trust.

In [6]:
print('top_k  lookback  skip   gross Sharpe    HAC t')
for tk, lb, sk, s, t in R['param_grid']:
    star = '  <- published' if (tk, lb, sk) == (3, 252, 21) else ''
    print('%5d %9d %5d %13.3f %8.2f%s' % (tk, lb, sk, s, t, star))
print('\nlargest |t| anywhere in the neighbourhood: %.2f'
      % max(abs(t) for *_, t in R['param_grid']))

top_k  lookback  skip   gross Sharpe    HAC t
    2       252    21         0.055     0.31
    3       252    21         0.099     0.55  <- published
    4       252    21         0.072     0.39
    5       252    21        -0.006    -0.02
    3       126    21         0.020     0.11
    3       252     0         0.046     0.25
    3        63     5         0.003     0.02

largest |t| anywhere in the neighbourhood: 0.55


The published setting happens to be the *best* of the seven — and its *t* is **+0.55**. That is the useful way to read a robustness grid on a dead strategy: not 'the result survives perturbation' but 'there was no setting worth reaching for in the first place'.

> ⚠️ Note what this check can and cannot buy. Seven points around one convention is not a multiple-testing correction, and if any cell here had printed *t* = 2.4 it would have been a **search statistic**, not a discovery. It is reported because a reader is entitled to know the headline number was not the survivor of a sweep.

## 5. Bootstrap Sharpe CIs (2,000 draws, 21-day fixed circular blocks)

In [7]:
print('speed        gross   95% CI                net    95% CI               share<0')
for i, lab in enumerate(R['labels']):
    g, n = R['ci_gross'][i], R['ci_net'][i]
    print('%-10s %+7.3f  [%+.3f, %+.3f]  %+7.3f  [%+.3f, %+.3f] %8.1f%%'
          % (lab, R['sharpe_gross'][i], g[0], g[1],
             R['sharpe_net'][i], n[0], n[1], R['ci_net_negshare'][i]))
print('\nevery interval straddles zero, gross and net, at every speed.')

speed        gross   95% CI                net    95% CI               share<0
daily       +0.121  [-0.221, +0.473]   -0.096  [-0.431, +0.252]     71.2%
weekly      +0.105  [-0.232, +0.457]   -0.003  [-0.337, +0.348]     51.4%
monthly     +0.099  [-0.252, +0.449]   +0.037  [-0.313, +0.388]     41.4%
quarterly   +0.013  [-0.319, +0.346]   -0.034  [-0.366, +0.297]     58.8%

every interval straddles zero, gross and net, at every speed.


## 6. Paired speed races

Same tape, same signal, only the clock differs — so the daily return difference is a clean paired comparison and the HAC *t* is the Jobson-Korkie return-difference test in Newey-West form.

In [8]:
print('race                   gross gap (t)        net gap (t)')
for name, gg, gt, ng, nt in R['races']:
    print('%-20s %+8.3f (%+5.2f)   %+8.3f (%+5.2f)' % (name, gg, gt, ng, nt))
print('\nnot one race clears |t| = 2 -- including the one the study is built to find')
print('(monthly beats daily NET at only t = %+.2f).' % R['races'][0][4])

race                   gross gap (t)        net gap (t)
daily - monthly        +0.012 (+0.18)     -0.144 (-1.90)
daily - quarterly      +0.098 (+0.90)     -0.072 (-0.67)
monthly - quarterly    +0.086 (+1.02)     +0.071 (+0.84)

not one race clears |t| = 2 -- including the one the study is built to find
(monthly beats daily NET at only t = -1.90).


## 7. Era cut (split 2013-01-01)

A budget that means anything should keep its shape across eras. This one does not.

In [9]:
print('speed        1999-2012 Sgross  break-even   2013-2026 Sgross  break-even')
for i, lab in enumerate(R['labels']):
    print('%-10s %+16.3f %10.2f bps %+17.3f %10.2f bps'
          % (lab, R['era_early_gross'][i], R['era_early_be'][i],
             R['era_late_gross'][i], R['era_late_be'][i]))
print('\nmonthly: the roomiest arm early (%.1f bps), NEGATIVE late (%.1f bps).'
      % (R['era_early_be'][2], R['era_late_be'][2]))
print('daily:   the tightest arm early (%.1f bps), the roomiest late (%.1f bps).'
      % (R['era_early_be'][0], R['era_late_be'][0]))

speed        1999-2012 Sgross  break-even   2013-2026 Sgross  break-even
daily                +0.035       0.31 bps            +0.154       3.50 bps
weekly               +0.103       4.92 bps            +0.047       1.24 bps
monthly              +0.137      15.86 bps            +0.014      -1.69 bps
quarterly            +0.013      -3.04 bps            -0.022     -11.07 bps

monthly: the roomiest arm early (15.9 bps), NEGATIVE late (-1.7 bps).
daily:   the tightest arm early (0.3 bps), the roomiest late (3.5 bps).


> 💡 **In plain words** — the 'best' rebalance frequency swapped places between the two halves of the sample. That is what a noise estimate looks like.

## 8. The long-only beta check

The most instructive table in the study. Drop the short leg, keep the ranking, and every speed clears *t* = 2 against cash. That is beta: a book that is always 100% in equities is being measured against T-bills.

The control is the **same eleven sectors equal-weighted, run through the same engine, on the same rebalance clock, with the same one-day lag, paying the same 5 bps on its own traded notional**. Cost symmetry is not a detail here — the sleeve turns over ~20× more than the control, so a frictionless benchmark would hand the sleeve's entire commission back as fake underperformance. (It did: see the warning below.) Both the cost-free and the net alpha are shown.

In [10]:
hdr = 'speed      Snet   t vs cash   EW-11   alphaG (t)        alphaN (t)      turn  turnEW'
print(hdr); print('-'*len(hdr))
for i, lab in enumerate(R['labels']):
    print('%-9s %+6.3f %+10.2f %8.3f %8.2f%% (%+5.2f) %8.2f%% (%+5.2f) %6.1fx %6.1fx'
          % (lab, R['lo_sharpe'][i], R['lo_t_cash'][i], R['lo_sharpe_ew'][i],
             R['lo_alpha_gross'][i], R['lo_t_ew_gross'][i],
             R['lo_alpha'][i], R['lo_t_ew'][i],
             R['lo_turn'][i], R['lo_turn_ew'][i]))
print('\nracing a fully-invested sleeve against CASH measures the equity market;')
print('racing it against a FRICTIONLESS benchmark measures its own commission.')

speed      Snet   t vs cash   EW-11   alphaG (t)        alphaN (t)      turn  turnEW
------------------------------------------------------------------------------------
daily     +0.524      +2.56    0.556     0.67% (+0.35)    -0.62% (-0.33)   27.2x    1.5x
weekly    +0.546      +2.65    0.555     0.40% (+0.21)    -0.19% (-0.10)   12.4x    0.7x
monthly   +0.539      +2.62    0.551     0.09% (+0.05)    -0.15% (-0.08)    5.2x    0.4x
quarterly +0.535      +2.62    0.584    -0.31% (-0.17)    -0.44% (-0.24)    2.8x    0.2x

racing a fully-invested sleeve against CASH measures the equity market;
racing it against a FRICTIONLESS benchmark measures its own commission.


> ⚠️ **An audit note against this study's own first draft.** The published cut of this table benchmarked the costed sleeve against a frictionless daily-rebalanced equal-weight average and reported selection alpha of −0.4% to −0.9%/yr, concluding that momentum 'loses to equal weighting at every speed'. With the benchmark paying its own way that shortfall largely disappears at the fast clocks — the gross alpha is +0.67%/yr at daily. The *conclusion* survives (nothing here is remotely significant; the largest |*t*| against the control is 0.35), but the original reason for it was an artefact. A one-sided cost ledger is the most common way a backtest lies in the direction of its own thesis — including a sceptical one.

## 9. Live synthetic control — the harness is unbiased

**Synthetic, not the real tape.** A planted panel whose expected returns decay with a ~63-day half-life: the ladder must recover a positive gross return that *falls* as the clock slows. A null panel (market factor + idiosyncratic noise): the ladder must go quiet at every speed.

In [11]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))
import numpy as np
from turnover_budget import data, strategy as st
p1, c1, _ = data.synthetic_panel(n_assets=8, n_years=12, signal_strength=1.0, seed=940)
d1 = st.synthetic_detect(p1, c1, top_k=2)
print('SYNTHETIC planted: gross Sharpe / HAC t by speed')
for f in ('D','W','M','Q'):
    print('   %-9s %+7.2f  (t %+6.2f)  turnover %5.1fx'
          % (st.FREQ_LABEL[f], d1['sharpe_gross'][f], d1['t_gross'][f],
             d1['ann_turnover'][f]))
print('   daily minus quarterly gross return: %+.2f pp/yr'
      % (d1['daily_minus_quarterly_gross']*100))
nulls = []
for s in range(5):
    pn, cn, _ = data.synthetic_panel(n_assets=8, n_years=12, signal_strength=0.0, seed=940+s)
    nulls.append(st.synthetic_detect(pn, cn, top_k=2)['sharpe_gross']['M'])
nulls = np.array(nulls)
print('\nSYNTHETIC null x5 (monthly gross Sharpe): mean %+.3f  sd %.3f  max|.| %.3f'
      % (nulls.mean(), nulls.std(ddof=1), np.abs(nulls).max()))

SYNTHETIC planted: gross Sharpe / HAC t by speed
   daily       +4.61  (t +11.15)  turnover   8.8x
   weekly      +4.41  (t +10.68)  turnover   5.2x
   monthly     +3.86  (t  +9.37)  turnover   3.5x
   quarterly   +3.13  (t  +7.55)  turnover   2.7x
   daily minus quarterly gross return: +11.69 pp/yr



SYNTHETIC null x5 (monthly gross Sharpe): mean -0.129  sd 0.157  max|.| 0.296


On the full-size synthetic panel used in `docs/results.md` the planted gross Sharpes are [5.78, 5.53, 4.87, 3.44] for daily/weekly/monthly/quarterly (HAC *t* = [17.7, 16.9, 15.0, 10.6]), with the daily arm earning **+13.9 pp/yr** more gross than the quarterly one; the null panel gives [-0.03, -0.135, -0.154, 0.025] with every |*t*| ≤ 0.67, and across six seeds the monthly gross Sharpe has mean -0.048 (sd 0.232). The detector fires on a planted effect and stays silent on the null — the flat real-tape result is a property of **sector momentum**, not of the engine.

## Verdict

- **Signal — None.** Best gross excess-of-cash HAC *t* across all four speeds: **+0.67** over 6,668 days. Gross Sharpes [0.121, 0.105, 0.099, 0.013] with bootstrap CIs that all straddle zero; the quarterly arm is +0.013, and the whole parameter neighbourhood tops out at *t* = +0.55. No paired speed race clears |*t*| = 2 (the best, monthly-beats-daily net, is *t* = -1.90). The long-only arm's *t* ≈ +2.6 against cash is equity beta: against a cost-matched, same-clock equal-weight-11 control its selection alpha is +0.67%/yr gross and -0.62%/yr net at the daily clock, with every |*t*| ≤ 0.35. Survivorship is small but named: XLRE (2015) and XLC (2018) are in the panel because GICS later carved them out.
- **Tradability — Mirage.** Break-even costs of 2.5 / 4.8 / 10.1 / -3.2 bps per unit traded notional — a budget for a statistically zero return. At 5 bps and 40 bps borrow three of four speeds are net-negative; at zero borrow the best arm is still only +0.062. The frequency ranking inverts *below* 1 bp, so 'which speed is best' is settled by an execution assumption, and the era cut swaps the winner outright.
- **The transferable result.** A daily clock on an eleven-name cross-section trades **31× NAV a year** (92% of it genuine rank churn) against a monthly clock's 5.8×, so it needs roughly five times the gross alpha to break even. Use the break-even column as the price of admission for any sleeve — and check the *gross* number clears it before admiring the net one.